# Customer Churn Prediction

## Feature Engineering

### Author: Aaron Thomas

## Table of Contents

<ol>
    <li>Introduction</li>
        <ol>
            <li>Overview</li>
            <li>Dataset Description</li>
        </ol>
    <li>Data Loading</li>
        <ol>
            <li>Importing Libraries</li>
            <li>Loading the Dataset</li>
            <li>Initial Data Inspection</li>
        </ol>
    <li>Feature Engineering</li>
        <ol>
            <li>Dropping Irrelevant Columns</li>
            <li>Change `TotalCharges` to Numeric Data Type</li>
            <li>Encoding Categorical Variables</li>
            <li>Creating New Features</li>
        </ol>
    <li>Train-Test Split</li>
        <ol>
            <li>Splitting the Data</li>
            <li>Verifying the Split</li>
            <li>Feature Scaling</li>
        </ol>
    <li>Handling Class Imbalance</li>
        <ol>
            <li>Applying SMOTE</li>
            <li>Verifying the Class Balance</li>
        </ol>
    <li>Saving the Processed Data</li>
        <ol>
            <li>Exporting to CSV</li>
        </ol>
    <li>Conclusion</li>
        <ol>
            <li>Summary</li>
            <li>Next Steps</li>
        </ol>
</ol>

## 1) Introduction

### 1.1) Overview

The purpose of this notebook is to perform feature engineering on the customer churn dataset in preparation for building a predictive model. Feature engineering involves transforming raw data into features that better represent the underlying problem to the predictive models, resulting in improved model performance.

### 1.2) Dataset Description

The customer churn dataset contains information about customers of a telecommunications company, including their demographic details, account information, and whether they have churned (i.e., stopped using the company's services). The dataset includes features that include:

- `customerID`: Unique identifier for each customer.
- `gender`: Gender
- `SeniorCitizen`: Whether the customer is a senior citizen (1, 0)
- `Partner`: Whether the customer has a partner (Yes, No)
- `Dependents`: Whether the customer has dependents (Yes, No)
- `tenure`: Number of months the customer has stayed with the company
- `PhoneService`: Whether the customer has phone service (Yes, No)
- `MultipleLines`: Whether the customer has multiple lines (Yes, No, No phone service)
- `InternetService`: Customer’s internet service provider (DSL, Fiber optic, No)
- `OnlineSecurity`: Whether the customer has online security (Yes, No, No internet service)
- `OnlineBackup`: Whether the customer has online backup (Yes, No, No internet service)
- `DeviceProtection`: Whether the customer has device protection (Yes, No, No internet service)
- `TechSupport`: Whether the customer has tech support (Yes, No, No internet service)
- `StreamingTV`: Whether the customer has streaming TV (Yes, No, No internet service
- `StreamingMovies`: Whether the customer has streaming movies (Yes, No, No internet service)
- `Contract`: The contract term of the customer (Month-to-month, One year, Two year)
- `PaperlessBilling`: Whether the customer has paperless billing (Yes, No)
- `PaymentMethod`: The customer’s payment method (Electronic check, Mailed check, Bank transfer (automatic), Credit card (automatic))
- `MonthlyCharges`: The amount charged to the customer monthly
- `TotalCharges`: The total amount charged to the customer
- `Churn`: Whether the customer churned (Yes or No)

## 2) Data Loading

### 2.1) Importing Libraries

The following libraries are imported for feature engineering and data manipulation:

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import joblib


### 2.2) Loading the Dataset

To load the dataset, the following code is used:

In [2]:
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### 2.3) Initial Data Inspection

In the EDA notebook, it was observed that there were no missing values in the dataset, but there were some inconsistencies in certain columns, as well as data types that needed to be corrected. To check for the data types, the following code is used:

In [3]:
df.dtypes

customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

Looking at the data types above, the following changes will be made:
- `customerID` will be dropped, as it is a unique identifier and does not provide any predictive value for the model.
- `TotalCharges` will be converted to a numeric data type, as it is currently an object due to the presence of spaces in some rows. This was observed during the EDA phase.
- One-hot encoding will be applied to the categorical variables to convert them into a format suitable for machine learning algorithms. They include `MultipleLines`, `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`, `Contract`, and `PaymentMethod`.
- The binary categorical variables `gender`, `Partner`, `Dependents`, `PhoneService`, and `PaperlessBilling` will be encoded as 0 and 1.
- New features will be created based on existing features, such as `AvgMonthlyCharges` (TotalCharges / tenure), `HasStreamingServices` (1 if StreamingTV or StreamingMovies is Yes, else 0), `NumAddOnServices` (count of additional services like OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport), and `IsNewCustomer` (1 if the customer has been with the company for less than 12 months, else 0).

## 3) Feature Engineering

### 3.1) Dropping Irrelevant Columns

For this step, the `customerID` column will be dropped from the dataset, as it is a unique identifier and does not provide any predictive value for the model. The following code is used to drop the column:

In [4]:
df.drop('customerID', axis=1, inplace=True)

### 3.2) Change `TotalCharges` to Numeric Data Type

In [5]:
df['TotalCharges'] = df['TotalCharges'].str.strip() # Remove leading/trailing spaces
df['TotalCharges'] = df['TotalCharges'].replace('', '0') # Replace empty strings with 0 (new customers)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])

### 3.3) Encoding Categorical Variables

The following code is used to convert the specified columns to binary values (0 and 1):

In [6]:
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})

*Note: The `SeniorCitizen` column is already in a binary format (1 for senior citizens and 0 for non-senior citizens), so no changes are needed for this column.*

The following code is used to convert the specified columns to categorical values:

In [7]:
cat_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity', 
            'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
            'StreamingMovies', 'Contract', 'PaymentMethod']

for col in cat_cols:
    df[col] = df[col].astype('category')

### 3.4) Creating New Features

To create the new features, the following code is used:

In [8]:
df['AvgMonthlyCharges'] = np.where(
    df['tenure'] == 0, # Avoid division by zero for new customers
    df['MonthlyCharges'], # If tenure is 0, use MonthlyCharges as AvgMonthlyCharges
    df['TotalCharges'] / df['tenure'] # Calculate average monthly charges for existing customers
)



df['HasStreamingServices'] = ((df['StreamingTV'] == 'Yes') | (df['StreamingMovies'] == 'Yes')).astype(int)
df['NumAddOnServices'] = (
    (df['OnlineSecurity'] == 'Yes').astype(int) + 
    (df['OnlineBackup'] == 'Yes').astype(int) + 
    (df['DeviceProtection'] == 'Yes').astype(int) + 
    (df['TechSupport'] == 'Yes').astype(int) + 
    (df['StreamingTV'] == 'Yes').astype(int) + 
    (df['StreamingMovies'] == 'Yes').astype(int)
)
df['IsNewCustomer'] = (df['tenure'] <= 12).astype(int) 

To verify that the new features have been created successfully, the following code is used:

In [9]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,AvgMonthlyCharges,HasStreamingServices,NumAddOnServices,IsNewCustomer
0,0,0,1,0,1,0,No phone service,DSL,No,Yes,...,Month-to-month,1,Electronic check,29.85,29.85,0,29.850000,0,1,1
1,1,0,0,0,34,1,No,DSL,Yes,No,...,One year,0,Mailed check,56.95,1889.50,0,55.573529,0,2,0
2,1,0,0,0,2,1,No,DSL,Yes,Yes,...,Month-to-month,1,Mailed check,53.85,108.15,1,54.075000,0,2,1
3,1,0,0,0,45,0,No phone service,DSL,Yes,No,...,One year,0,Bank transfer (automatic),42.30,1840.75,0,40.905556,0,3,0
4,0,0,0,0,2,1,No,Fiber optic,No,No,...,Month-to-month,1,Electronic check,70.70,151.65,1,75.825000,0,0,1


To one-hot encode the categorical columns, the following code is used:

In [10]:
df = pd.get_dummies(df, columns=cat_cols, drop_first=True) # drop_first = True to avoid dummy variable trap

## 4) Train-Test Split

The purpose of the train-test split is to divide the dataset into two subsets: one for training the machine learning model and another for testing its performance. This ensures that the model is evaluated on unseen data, as well as to prevent overfitting. The following code is used to perform the train-test split:

### 4.1) Splitting the Data

In [11]:
X = df.drop('Churn', axis=1)
y = df['Churn']

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y)

*Note: This was split into 80% training data and 20% testing data, with a random state of 42 for reproducibility, and stratification based on the target variable `Churn` to maintain the same class distribution in both sets.*

In [13]:
X_train = X_train.copy()
X_test = X_test.copy()

The purpose of the .copy() method above is to create a copy of the training and testing feature sets (X_train and X_test) to ensure that any modifications made to these DataFrames do not affect the original DataFrames. This is important because some operations, such as scaling or encoding, can modify the DataFrame in place, and using .copy() helps to avoid unintended side effects on the original data.

### 4.2) Verifying the Split

To verify that the train-test split was successful and that the class distribution of the target variable `Churn` is maintained in both sets, the following code is used:

In [14]:
# Verify the split
print("Training set class distribution:")
print(y_train.value_counts(normalize=True))
print("\nTesting set class distribution:")
print(y_test.value_counts(normalize=True))

Training set class distribution:
Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64

Testing set class distribution:
Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64


### 4.3) Feature Scaling

The purpose of this step is to scale the numerical features to ensure that they are on the same scale, which can improve the performance of machine learning algorithms. The `tenure`, `MonthlyCharges`, `TotalCharges`, and `AvgMonthlyCharges` columns will be scaled using StandardScaler. The following code is used for feature scaling:

In [15]:
scaler = StandardScaler()
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlyCharges']
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

## 5) Handling Class Imbalance

In the EDA notebook, a visualization of the target variable `Churn` revealed that there was a class imbalance, with 73.5% of the customers not churning and 26.5% of the customers churning. To address this, SMOTE (Synthetic Minority Over-sampling Technique) will be applied to the training data to create synthetic samples of the churned customers (the minority class) to balance the class distribution. The following code is used to apply SMOTE:

In [16]:
smote = SMOTE(random_state=42) # initialize SMOTE
X_train, y_train = smote.fit_resample(X_train, y_train) # Apply SMOTE to the training data

In [17]:
print("Class Distribution after SMOTE:")
print(pd.Series(y_train).value_counts())

Class Distribution after SMOTE:
Churn
0    4139
1    4139
Name: count, dtype: int64


After applying SMOTE, the results above show that the class distribution is now balanced.

## 6) Saving the Processed Data

After cleaning and performing feature engineering techniques on the dataset, the processed training and testing data will be saved to CSV files for use in the modeling notebook. The following code is used to save the processed data:

In [18]:
X_train.to_csv('../data/processed/X_train.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

In addition, the fitted StandardScaler object will also be saved using joblib, so that it can be used for making predictions in the future. The following code is used to save the StandardScaler object:

In [20]:
joblib.dump(scaler, '../models/standard_scaler.joblib')

['../models/standard_scaler.joblib']

## 7) Conclusion

### 7.1) Summary

Overall, this notebook performed feature engineering on the customer churn dataset, which included:
- Dropping irrelevant columns
- Converting data types
- Encoding categorical variables
- Creating new features
- Scaling numerical features
- Splitting the data into training and testing sets
- Handling class imbalance using SMOTE

After processing, the dataset now contains 35 features, and the processed data was saved to CSV files for use in the modeling phase of the project.

### 7.2) Next Steps

The next steps in this project will involve building and evaluating machine learning models to predict customer churn using the processed data. This will be done in a separate notebook dedicated to modeling, where various algorithms will be tested and their performance compared to select the best model for deployment.